<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [ ]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, LabelEncoder, OrdinalEncoder
from google.colab import files

## Implementation of PlottingMethods Class

In [ ]:
class PlottingMethods:
    """
    A modular utility class designed to generate granular charts (Bar, Pie, Histograms)
    and return them as HTML-wrapped strings or display them for versatile integration.
    """
    
    @staticmethod
    def generate_bar_chart(df, x_col, y_col, title="Bar Chart", return_html=False):
        """
        Generates a Plotly Bar Chart.
        """
        fig = px.bar(df, x=x_col, y=y_col, title=title, template="plotly_white")
        if return_html:
            return fig.to_html(full_html=False, include_plotlyjs='cdn')
        fig.show()
        return fig

    @staticmethod
    def generate_pie_chart(df, names_col, values_col=None, title="Pie Chart", return_html=False):
        """
        Generates a Plotly Pie Chart.
        """
        fig = px.pie(df, names=names_col, values=values_col, title=title, template="plotly_white")
        if return_html:
            return fig.to_html(full_html=False, include_plotlyjs='cdn')
        fig.show()
        return fig

    @staticmethod
    def generate_histogram(df, x_col, n_bins=None, title="Histogram", return_html=False):
        """
        Generates a Plotly Histogram.
        """
        fig = px.histogram(df, x=x_col, nbins=n_bins, title=title, template="plotly_white")
        if return_html:
            return fig.to_html(full_html=False, include_plotlyjs='cdn')
        fig.show()
        return fig

## Implementation of DataInspector Class

In [ ]:
class DataInspector:
    """
    An end-to-end interactive toolkit class for automated CSV ingestion, sanitization,
    structural cleaning, outlier management, normalization, and complex statistical analysis.
    """
    def __init__(self, df=None):
        self.df = df.copy() if df is not None else None
        
    def upload_data(self):
        """
        Triggers a local files upload mechanism via Google Colab, handles CSV files,
        and applies garbage string sanitization instantly.
        """
        print("Please select your CSV file to upload:")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return
        
        filename = list(uploaded.keys())[0]
        # Sanitize common garbage strings on ingestion
        garbage_values = ['?', 'n/a', 'NULL', ' ', 'N/A', 'null', 'nan', 'NaN']
        self.df = pd.read_csv(io.BytesIO(uploaded[filename]), na_values=garbage_values)
        print(f" Successfully ingested {filename}. Formatted into DataFrame.")
        self.auto_type_correction()
        
    def auto_type_correction(self):
        """
        Force-converts object columns to numeric types if the conversion 
        does not yield an completely null column.
        """
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                # Try conversion
                converted = pd.to_numeric(self.df[col], errors='coerce')
                if not converted.isna().all():
                    self.df[col] = converted
(        print("Auto-type correction pass finished.")
        
    def display_summary(self):
        """
        Provides a structural high-level view of row/column metrics, preview rows,
        and specific lists of column classifications.
        """
        if self.df is None:
            print("No data active inside the inspector.")
            return
        
        print("="*50)
        print(f"DATASET METRICS SUMMARY")
        print("="*50)
        print(f"Total Rows: {self.df.shape[0]}")
        print(f"Total Columns: {self.df.shape[1]}")
        
        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        
        print(f"\nNumerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
        
        print("\n--- First 20 Rows Preview ---")
        display(self.df.head(20))
        
    def handle_missing_values(self, column, strategy='mean', constant_value=None):
        """
        Imputes missing items cleanly across specified columns via specified strategies.
        Strategies: 'mean', 'median', 'mode', 'constant'
        """
        if self.df is None or column not in self.df.columns: return
        
        if strategy == 'mean':
            val = self.df[column].mean()
            self.df[column].fillna(val, inplace=True)
        elif strategy == 'median':
            val = self.df[column].median()
            self.df[column].fillna(val, inplace=True)
        elif strategy == 'mode':
            val = self.df[column].mode()[0] if not self.df[column].mode().empty else np.nan
            self.df[column].fillna(val, inplace=True)
        elif strategy == 'constant':
            self.df[column].fillna(constant_value, inplace=True)
        print(f"Imputation handled for column '{column}' via strategy: {strategy}.")
        
    def remove_duplicates(self):
        """
        Prunes all exact multi-row structural layout duplications.
        """
        if self.df is None: return
        initial = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial - self.df.shape[0]} perfect exact matching rows.")
        
    def handle_outliers(self, column, action='flag'):
        """
        Advanced IQR detection engine. Flags anomalies or automatically drops matching rows.
        Actions: 'flag', 'delete'
        """
        if self.df is None or column not in self.df.columns: return self.df
        if not np.issubdtype(self.df[column].dtype, np.number):
            print("Outlier analysis is reserved strictly for numeric types.")
            return
            
        Q1 = self.df[column].quantile(0.25)
        Q3 = self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_mask = (self.df[column] < lower_bound) | (self.df[column] > upper_bound)
        
        if action == 'flag':
            self.df[f'{column}_outlier'] = outliers_mask.astype(int)
            print(f"Flagged {outliers_mask.sum()} rows as outliers in '{column}_outlier'.")
        elif action == 'delete':
            self.df = self.df[~outliers_mask]
            print(f"Dropped {outliers_mask.sum()} outlier rows based on column '{column}'.")
            
    def delete_rows(self, indices_str):
        """
        Prunes rows dynamically via a comma-separated string index list.
        """
        if self.df is None: return
        try:
            indices = [int(i.strip()) for i in indices_str.split(',') if i.strip()]
            self.df.drop(index=indices, inplace=True, errors='ignore')
            print(f"Pruned rows matching indices: {indices}")
        except Exception as e:
            print(f"Parsing issue during row deletion: {e}")
            
    def delete_columns(self, cols_str):
        """
        Prunes target columns dynamically via a comma-separated column string sequence.
        """
        if self.df is None: return
        cols = [c.strip() for c in cols_str.split(',') if c.strip()]
        self.df.drop(columns=cols, inplace=True, errors='ignore')
        print(f"Pruned column elements matching: {cols}")
        
    def extract_normalized_numeric_data(self, columns, method='standard'):
        """
        Extracts scaled versions of numeric attributes via 'minmax', 'standard', or 'robust'.
        """
        if self.df is None: return pd.DataFrame()
        sub_df = self.df[columns].copy()
        
        if method == 'minmax':
            scaler = MinMaxScaler()
        elif method == 'standard':
            scaler = StandardScaler()
        elif method == 'robust':
            scaler = RobustScaler()
        else:
            raise ValueError("Unknown method flag. Pick 'minmax', 'standard', or 'robust'.")
            
        scaled_values = scaler.fit_transform(sub_df.fillna(sub_df.median()))
        return pd.DataFrame(scaled_values, columns=[f"{c}_{method}" for c in columns], index=self.df.index)
        
    def extract_normalized_categorical_data(self, columns, method='onehot'):
        """
        Encodes complex string variables via 'onehot', 'ordinal', or 'uniform'.
        """
        if self.df is None: return pd.DataFrame()
        sub_df = self.df[columns].copy().astype(str)
        
        if method == 'onehot':
            return pd.get_dummies(sub_df, columns=columns, prefix=[f"{c}_oh" for c in columns])
        elif method == 'ordinal':
            oe = OrdinalEncoder()
            encoded = oe.fit_transform(sub_df)
            return pd.DataFrame(encoded, columns=[f"{c}_ord" for c in columns], index=self.df.index)
        elif method == 'uniform':
            # Ordinal scaling normalized between 0 and 1
            oe = OrdinalEncoder()
            encoded = oe.fit_transform(sub_df)
            for i in range(encoded.shape[1]):
                max_val = encoded[:, i].max()
                if max_val > 0:
                    encoded[:, i] = encoded[:, i] / max_val
            return pd.DataFrame(encoded, columns=[f"{c}_uni" for c in columns], index=self.df.index)
            
    def get_unified_dataset(self, num_cols, cat_cols, num_method='standard', cat_method='onehot'):
        """
        Returns a single integrated DataFrame layout binding original numerical keys alongside encoded attributes.
        """
        scaled_num = self.extract_normalized_numeric_data(num_cols, method=num_method)
        encoded_cat = self.extract_normalized_categorical_data(cat_cols, method=cat_method)
        return pd.concat([self.df[num_cols], scaled_num, encoded_cat], axis=1)
        
    def generate_univariate_subplots(self, column):
        """
        Generates an active 3-panel layout dashboard chart tracking single metrics via Box, Scatter, and Histograms.
        """
        if self.df is None or column not in self.df.columns: return
        
        fig = make_subplots(
            rows=1, cols=3, 
            subplot_titles=("Violin/Box Distribution", "Index Scatter Plot", "Histogram Plot")
        )
        
        clean_data = self.df[column].dropna()
        
        fig.add_trace(go.Violin(y=clean_data, name=column, box_visible=True, points='all'), row=1, col=1)
        fig.add_trace(go.Scatter(x=clean_data.index, y=clean_data, mode='markers', name='Value vs Index'), row=1, col=2)
        fig.add_trace(go.Histogram(x=clean_data, name='Frequency count'), row=1, col=3)
        
        fig.update_layout(title_text=f"Univariate Analysis Dashboard for Parameter: {column}", template="plotly_white", showlegend=False)
        fig.show()
        
    def plot_relationship(self, col1, col2):
        """
        Dynamically picks optimal charting mechanics based on underlying attribute profiles.
        """
        if self.df is None or col1 not in self.df.columns or col2 not in self.df.columns: return
        
        is_num1 = np.issubdtype(self.df[col1].dtype, np.number)
        is_num2 = np.issubdtype(self.df[col2].dtype, np.number)
        
        if is_num1 and is_num2:
            # Num-Num
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Scatter Chart (Num-Num) with OLS Line: {col1} vs {col2}", template="plotly_white")
            fig.show()
        elif not is_num1 and not is_num2:
            # Cat-Cat
            ct = pd.crosstab(self.df[col1], self.df[col2]).reset_index()
            df_melt = ct.melt(id_vars=col1)
            fig = px.bar(df_melt, x=col1, y='value', color=col2, barmode='group', title=f"Grouped Bar Chart (Cat-Cat): {col1} vs {col2}", template="plotly_white")
            fig.show()
        else:
            # Cat-Num
            cat = col1 if not is_num1 else col2
            num = col2 if not is_num1 else col1
            fig = px.box(self.df, x=cat, y=num, points="all", title=f"Box Plot with Data Points (Cat-Num): {cat} vs {num}", template="plotly_white")
            fig.show()
            
    def plot_categorical_frequency(self, column):
        """
        Builds clean structured vertical summaries showcasing counts alongside explicit percentage distributions.
        """
        if self.df is None or column not in self.df.columns: return
        
        counts = self.df[column].value_counts(dropna=True).reset_index()
        counts.columns = [column, 'Count']
        total = counts['Count'].sum()
        counts['Percentage'] = (counts['Count'] / total * 100).round(2).astype(str) + '%'
        
        fig = px.bar(counts, x=column, y='Count', text='Percentage', title=f"Categorical Frequencies & Distributions: {column}", template="plotly_white")
        fig.update_traces(textposition='outside')
        fig.show()
        
    def plot_all_associations_heatmap(self):
        """
        Advanced statistical evaluation engine parsing Pearson's r, Cramér's V, and ANOVA Eta matrices
        across asymmetric column relationships.
        """
        if self.df is None: return
        
        cols = self.df.columns.tolist()
        n = len(cols)
        matrix = np.zeros((n, n))
        
        for i in range(n):
            for j in range(n):
                c1, c2 = cols[i], cols[j]
                
                # Handle perfect identity
                if i == j:
                    matrix[i, j] = 1.0
                    continue
                    
                temp_df = self.df[[c1, c2]].dropna()
                if temp_df.empty:
                    matrix[i, j] = 0.0
                    continue
                    
                is_num1 = np.issubdtype(self.df[c1].dtype, np.number)
                is_num2 = np.issubdtype(self.df[c2].dtype, np.number)
                
                if is_num1 and is_num2:
                    # Pearson's r
                    r, _ = stats.pearsonr(temp_df[c1], temp_df[c2])
                    matrix[i, j] = np.nan_to_num(r)
                elif not is_num1 and not is_num2:
                    # Cramér's V
                    confusion_matrix = pd.crosstab(temp_df[c1], temp_df[c2])
                    if confusion_matrix.size == 0 or confusion_matrix.shape[0] <= 1 or confusion_matrix.shape[1] <= 1:
                        matrix[i, j] = 0.0
                    else:
                        chi2 = stats.chi2_contingency(confusion_matrix)[0]
                        n_obs = confusion_matrix.sum().sum()
                        phi2 = chi2 / n_obs
                        r_shape, c_shape = confusion_matrix.shape
                        phi2corr = max(0, phi2 - ((c_shape-1)*(r_shape-1))/(n_obs-1))
                        rcorr = r_shape - ((r_shape-1)**2)/(n_obs-1)
                        ccorr = c_shape - ((c_shape-1)**2)/(n_obs-1)
                        if min((ccorr-1), (rcorr-1)) <= 0:
                            matrix[i, j] = 0.0
                        else:
                            matrix[i, j] = np.sqrt(phi2corr / min((ccorr-1), (rcorr-1)))
                else:
                    # Mixed Variable Types: Eta Squared calculation (ANOVA framework)
                    num_col = c1 if is_num1 else c2
                    cat_col = c2 if is_num1 else c1
                    
                    groups = [group[num_col].values for name, group in temp_df.groupby(cat_col)]
                    if len(groups) <= 1 or sum(len(g) for g in groups) <= len(groups):
                        matrix[i, j] = 0.0
                    else:
                        f_val, _ = stats.f_oneway(*groups)
                        # Approximate correlation magnitude from F statistics
                        df_between = len(groups) - 1
                        df_within = sum(len(g) for g in groups) - len(groups)
                        if (f_val * df_between + df_within) == 0:
                            matrix[i, j] = 0.0
                        else:
                            eta2 = (f_val * df_between) / (f_val * df_between + df_within)
                            matrix[i, j] = np.sqrt(eta2)
                            
        fig = go.Figure(data=go.Heatmap(
            z=matrix, x=cols, y=cols,
            colorscale='Viridis', zmin=-1.0, zmax=1.0
        ))
        fig.update_layout(title='Unified Mixed-Type Association Heatmap (Pearson / Cramer\'s V / Eta)', template='plotly_white')
        fig.show()

## Real-World Comprehensive Verification Flow (Titanic Dataset Simulation)

In [ ]:
# Download and load Titanic data directly to showcase and verify the operational logic
print("Downloading standard Titanic dataset...")
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
raw_df = pd.read_csv(titanic_url)

# Initialize engine
inspector = DataInspector(raw_df)

# 1. Display raw ingestion summary metrics
inspector.display_summary()

# 2. Handle missing data values across targets
inspector.handle_missing_values('Age', strategy='median')
inspector.handle_missing_values('Embarked', strategy='mode')

# 3. Manage duplicate entries
inspector.remove_duplicates()

# 4. Analyze and filter outliers from quantitative values
inspector.handle_outliers('Fare', action='flag')

# 5. Check univariate statistical attributes
inspector.generate_univariate_subplots('Age')

# 6. Evaluate specific bivariate relationships
inspector.plot_relationship('Pclass', 'Fare') # Cat-Num
inspector.plot_relationship('Sex', 'Embarked') # Cat-Cat

# 7. Generate precise frequency insights with tags
inspector.plot_categorical_frequency('Pclass')

# 8. Create association matrix map metrics
inspector.plot_all_associations_heatmap()

# 9. Output complete combined structured datasets
unified_df = inspector.get_unified_dataset(num_cols=['Age', 'Fare'], cat_cols=['Sex', 'Embarked'], num_method='minmax', cat_method='onehot')
print("\n--- Combined Engineered Unified Dataset Layout Preview ---")
display(unified_df.head(10))